# manchuBERT as Masked Language Model

-   clarify the transliteration system (Abkai or Norman?)
-   reference: https://huggingface.co/seemdog/manchuBERT

## manchuBERT: Model and Vocabularies

Model page: https://huggingface.co/seemdog/manchuBERT

In [36]:
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM
import torch

In [37]:
# Use a pipeline as a high-level helper
pipe = pipeline("fill-mask", model="seemdog/manchuBERT")

Device set to use cpu

In [38]:
# Load vocabularies and model from huggingface and cache
tokenizer = AutoTokenizer.from_pretrained("seemdog/manchuBERT")
model = AutoModelForMaskedLM.from_pretrained("seemdog/manchuBERT")

### Inspect the Pre-trained Model

In [34]:
print("--- Tokenizer Attributes ---")
print(f"Number of tokens (vocabulary size): {tokenizer.vocab_size}")

print("--- Model Attributes ---")
if hasattr(model.config, 'hidden_size'):
    print(f"Embedding vector dimension (hidden size): {model.config.hidden_size}")
elif hasattr(model.config, 'd_model'):
    print(f"Embedding vector dimension (d_model): {model.config.d_model}")
else:
    print("Could not determine embedding vector dimension from model config.")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

--- Tokenizer Attributes ---
Number of tokens (vocabulary size): 25000
--- Model Attributes ---
Embedding vector dimension (hidden size): 768
Total number of parameters: 105,267,880

## Generate

In [27]:
# Input a masked Manchu sentence then collect the top 5 predictions
def predict_masked(text, tokenizer, model, reference_text):
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)

    # collect logits
    mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
    logits = outputs.logits[0, mask_token_index, :]
    top_5_token_ids = torch.topk(logits, 5, dim=-1).indices[0].tolist()

    print(f"Input text:")
    print(text)
    print('-' * 30)
    print("Top 5 predictions:")
    predictions = []
    for token_id in top_5_token_ids:
        decoded_token = tokenizer.decode([token_id])
        print(f"    - {decoded_token}")
        predictions.append(decoded_token)
    print('-' * 30)
    print(f'Reference: {reference_text}')
    return predictions

In [28]:
# EN: this is a pretty good name
predict_masked('ere gebu umesi [MASK].', tokenizer, model, reference_text = 'ere gebu umesi sain')

Input text:
ere gebu umesi [MASK].
------------------------------
Top 5 predictions:
    - wesihun
    - ambula
    - .
    - sain
    - saikan
------------------------------
Reference: ere gebu umesi sain

['wesihun', 'ambula', '.', 'sain', 'saikan']

In [29]:
# EN: you need not be in a hurry
predict_masked('[MASK] ume ekxere.', tokenizer, model, reference_text= 'si ume ekxere')

Input text:
[MASK] ume ekxere.
------------------------------
Top 5 predictions:
    - enenggi
    - .
    - suwe
    - pangtung
    - adarame
------------------------------
Reference: si ume ekxere

['enenggi', '.', 'suwe', 'pangtung', 'adarame']

#### If they are using Abkai transliteration, they should have `qooha` rather than `cooha` (the Mollendorff-Norman Transliteration, use `c`)

In [31]:
# EN: so let us go together and have a look
predict_masked('uhei geneme emjergi [MASK] tuwaki', tokenizer, model, reference_text= 'uhei geneme emjergi sarxame tuwaki')

Input text:
uhei geneme emjergi [MASK] tuwaki
------------------------------
Top 5 predictions:
    - cooha
    - manggi
    - be
    - isinjiha
    - bihe
------------------------------
Reference: uhei geneme emjergi sarxame tuwaki

['cooha', 'manggi', 'be', 'isinjiha', 'bihe']

#### But they do use `v` instead of `ū` for the vowel

In [32]:
predict_masked('buta halan [MASK].', tokenizer, model, reference_text= 'buta halan akv')

Input text:
buta halan [MASK].
------------------------------
Top 5 predictions:
    - be
    - mukiyebure
    - akv
    - nadan
    - ilan
------------------------------
Reference: buta halan akv

['be', 'mukiyebure', 'akv', 'nadan', 'ilan']

#### `x` instead of `š`

In [33]:
predict_masked('tere erebe minde emu fengseku [MASK] ilha.', tokenizer, model, reference_text= 'tere erebe minde emu fengseku xunggiyada ilha.')

Input text:
tere erebe minde emu fengseku [MASK] ilha.
------------------------------
Top 5 predictions:
    - i
    - .
    - xanggiyan
    - orho
    - noho
------------------------------
Reference: tere erebe minde emu fengseku xunggiyada ilha.

['i', '.', 'xanggiyan', 'orho', 'noho']